# 1. Setup

## 1.1 Detección automática de las rutas de "input" y "working" de kaggle 

In [1]:
# ============================================================
# DETECCIÓN AUTOMÁTICA DE CSV EN KAGGLE INPUT
# ============================================================

from pathlib import Path
import shutil

INPUT_ROOT = Path("/kaggle/input")

print("CSV detectados:\n")

csv_files = list(INPUT_ROOT.rglob("*.csv"))

for f in csv_files:
    print(f)

# ============================================================
# COPIAR CSV A WORKING/data/raw
# ============================================================

RAW = Path("/kaggle/working/data/raw")
RAW.mkdir(parents=True, exist_ok=True)

for file in csv_files:
    shutil.copy(file, RAW / file.name)

print(f"\nCSV copiados correctamente: {len(csv_files)}")
print(f"Destino: {RAW}")

CSV detectados:

/kaggle/input/datasets/sity127uoc/m5-uoc/calendar.csv
/kaggle/input/datasets/sity127uoc/m5-uoc/sell_prices.csv
/kaggle/input/datasets/sity127uoc/m5-uoc/sales_train_validation.csv

CSV copiados correctamente: 3
Destino: /kaggle/working/data/raw


## 1.2. Crear estructura de carpetas en working

In [2]:
# ============================================================
# CREACIÓN DE ESTRUCTURA
# ============================================================

from pathlib import Path
import shutil

WORK = Path("/kaggle/working")

RAW = WORK / "data" / "raw"
FEATURES = WORK / "data" / "features"

SRC = WORK / "src"

SPARK_SRC = SRC / "spark"
MODELING_SRC = SRC / "modeling"
POWERBI_SRC = SRC / "powerbi"

MODELS = WORK / "models"

POST = WORK / "post_analysis"

LOGS = POST / "logs"
FIGURES = POST / "figures"
PREDICTIONS = POST / "predictions"

# ============================================================
# CREAR CARPETAS
# ============================================================

for p in [
    RAW,
    FEATURES,
    SRC,
    SPARK_SRC,
    MODELING_SRC,
    POWERBI_SRC,
    MODELS,
    POST,
    LOGS,
    FIGURES,
    PREDICTIONS
]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# COPIAR RAW FILES DESDE KAGGLE INPUT A /kaggle/working/data/raw
# ============================================================

INPUT_ROOT = Path("/kaggle/input")

csv_files = list(INPUT_ROOT.rglob("*.csv"))

for file in csv_files:
    shutil.copy(file, RAW / file.name)

print("Estructura preparada correctamente.")
print(f"CSV copiados en {RAW}: {len(csv_files)}")

for f in RAW.glob("*.csv"):
    print(f)

Estructura preparada correctamente.
CSV copiados en /kaggle/working/data/raw: 3
/kaggle/working/data/raw/calendar.csv
/kaggle/working/data/raw/sell_prices.csv
/kaggle/working/data/raw/sales_train_validation.csv


## 2. Spark dataload y build features

## 2.1 SparkSession

In [3]:
%%writefile /kaggle/working/src/spark/session.py
from pyspark.sql import SparkSession

def get_spark():
    return (
        SparkSession.builder
        .appName("M5_pipeline")
        .config("spark.driver.memory", "6g")
        .config("spark.sql.shuffle.partitions", "200")
        .getOrCreate()
    )

Overwriting /kaggle/working/src/spark/session.py


## 2.2 load_data.py

In [4]:
%%writefile /kaggle/working/src/spark/load_data.py
def load_data(spark, path="data/raw/"):

    sales = spark.read.csv(f"{path}/sales_train_validation.csv", header=True, inferSchema=True)
    calendar = spark.read.csv(f"{path}/calendar.csv", header=True, inferSchema=True)
    prices = spark.read.csv(f"{path}/sell_prices.csv", header=True, inferSchema=True)

    return sales, calendar, prices

Overwriting /kaggle/working/src/spark/load_data.py


## 2.3 build_features.py 33f (33 features). 

In [5]:
%%writefile /kaggle/working/src/spark/build_features.py
from pyspark.sql.functions import expr, col, lag, avg, stddev, broadcast, weekofyear
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel


def build_features(sales, calendar, prices):

    # --------------------------------------------------
    # 1. MELT / UNPIVOT sales_train_validation
    # --------------------------------------------------
    cols = sales.columns[6:]

    expr_str = "stack({}, {}) as (d, sales)".format(
        len(cols),
        ",".join([f"'{c}', {c}" for c in cols])
    )

    sales_long = sales.selectExpr(*sales.columns[:6], expr_str)

    # --------------------------------------------------
    # 2. Persist datasets reutilizados
    # --------------------------------------------------
    sales_long.persist(StorageLevel.MEMORY_AND_DISK)
    prices.persist(StorageLevel.MEMORY_AND_DISK)

    # --------------------------------------------------
    # 3. JOINS optimizados
    # --------------------------------------------------
    df = sales_long.join(
        broadcast(calendar),
        "d",
        "left"
    )

    df = df.join(
        prices,
        ["store_id", "item_id", "wm_yr_wk"],
        "left"
    )

    # --------------------------------------------------
    # 4. Tipos
    # --------------------------------------------------
    df = df.withColumn("date", col("date").cast("date"))

    # --------------------------------------------------
    # 5. Repartition antes de ventanas
    # --------------------------------------------------
    df = df.repartition("store_id", "item_id")

    # --------------------------------------------------
    # 6. Window por serie item-store
    # --------------------------------------------------
    window = Window.partitionBy(
        "item_id",
        "store_id"
    ).orderBy("date")

    # --------------------------------------------------
    # 7. Lags de ventas
    # --------------------------------------------------
    df = df.withColumn("lag_1", lag("sales", 1).over(window))
    df = df.withColumn("lag_7", lag("sales", 7).over(window))
    df = df.withColumn("lag_14", lag("sales", 14).over(window))
    df = df.withColumn("lag_28", lag("sales", 28).over(window))

    # --------------------------------------------------
    # 8. Rolling windows usando solo pasado
    # --------------------------------------------------
    rolling_7 = window.rowsBetween(-7, -1)
    rolling_14 = window.rowsBetween(-14, -1)
    rolling_28 = window.rowsBetween(-28, -1)

    df = df.withColumn("rolling_mean_7", avg("sales").over(rolling_7))
    df = df.withColumn("rolling_mean_14", avg("sales").over(rolling_14))
    df = df.withColumn("rolling_mean_28", avg("sales").over(rolling_28))

    df = df.withColumn("rolling_std_7", stddev("sales").over(rolling_7))
    df = df.withColumn("rolling_std_28", stddev("sales").over(rolling_28))

    df = df.withColumn("rolling_max_28", expr("max(sales)").over(rolling_28))
    df = df.withColumn("rolling_min_28", expr("min(sales)").over(rolling_28))

    # --------------------------------------------------
    # 9. Features de precio
    # --------------------------------------------------
    df = df.withColumn("price_lag_1", lag("sell_price", 1).over(window))

    df = df.withColumn(
        "price_diff_1",
        col("sell_price") - col("price_lag_1")
    )

    df = df.withColumn(
        "price_pct_change_1",
        expr("""
            CASE
                WHEN price_lag_1 IS NULL OR price_lag_1 = 0 THEN 0
                ELSE (sell_price - price_lag_1) / price_lag_1
            END
        """)
    )

    price_avg_window = Window.partitionBy("item_id", "store_id")

    df = df.withColumn(
        "price_mean_item_store",
        avg("sell_price").over(price_avg_window)
    )

    df = df.withColumn(
        "price_norm_item_store",
        expr("""
            CASE
                WHEN price_mean_item_store IS NULL OR price_mean_item_store = 0 THEN 0
                ELSE sell_price / price_mean_item_store
            END
        """)
    )

    # --------------------------------------------------
    # 10. Features calendario adicionales
    # --------------------------------------------------
    df = df.withColumn("weekofyear", weekofyear(col("date")))

    # --------------------------------------------------
    # 11. Limpieza SOLO de features numéricas nuevas
    # --------------------------------------------------
    numeric_features_to_fill = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28",
        "rolling_max_28",
        "rolling_min_28",
        "price_lag_1",
        "price_diff_1",
        "price_pct_change_1",
        "price_mean_item_store",
        "price_norm_item_store",
        "weekofyear"
    ]

    df = df.fillna(0, subset=numeric_features_to_fill)

    # --------------------------------------------------
    # 12. Resultado final
    # --------------------------------------------------
    return df

Overwriting /kaggle/working/src/spark/build_features.py


## 2.4 validate.py

In [6]:
%%writefile /kaggle/working/src/spark/validate.py
from pyspark.storagelevel import StorageLevel

def validate_data(sales, calendar, prices):

    # Cache temporal para no releer archivos varias veces
    sales.persist(StorageLevel.MEMORY_AND_DISK)
    calendar.persist(StorageLevel.MEMORY_AND_DISK)
    prices.persist(StorageLevel.MEMORY_AND_DISK)

    sales_rows = sales.count()
    calendar_rows = calendar.count()
    prices_rows = prices.count()

    print(f"Sales rows: {sales_rows}")
    print(f"Calendar rows: {calendar_rows}")
    print(f"Prices rows: {prices_rows}")

    # Duplicados
    sales_unique = sales.dropDuplicates().count()
    duplicates = sales_rows - sales_unique

    print(f"Sales sin duplicados: {sales_unique}")
    print(f"Duplicados detectados: {duplicates}")


Overwriting /kaggle/working/src/spark/validate.py


## 3 Modeling + Shap + Mint

## 3.1 run_modelo_global

In [7]:
%%writefile /kaggle/working/src/modeling/run_modelo_global.py
import pandas as pd
import lightgbm as lgb
import joblib
from pathlib import Path
import time
import gc
import re

start_total = time.time()

BASE_DIR = Path("/kaggle/working")
DATA_PATH = BASE_DIR / "data" / "features" / "m5_features"
MODEL_PATH = BASE_DIR / "models"
CHECKPOINT_PATH = MODEL_PATH / "checkpoints"

MODEL_PATH.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH.mkdir(parents=True, exist_ok=True)

stores = pd.read_parquet(
    DATA_PATH,
    columns=["store_id"]
)["store_id"].unique().tolist()

categorical_cols = [
    "store_id", "item_id", "dept_id", "cat_id", "state_id",
    "weekday", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2"
]

progress_file = CHECKPOINT_PATH / "training_progress.csv"
features_file = MODEL_PATH / "features_global.pkl"
categorical_file = MODEL_PATH / "categorical_cols_global.pkl"

model = None
features = None
categorical_used = None
completed_stores = []

# =========================
# RESUME FROM CHECKPOINT
# =========================

if progress_file.exists():
    progress_df = pd.read_csv(progress_file)
    completed_stores = progress_df["store_id"].tolist()

    if len(completed_stores) > 0:
        last_store = completed_stores[-1]
        checkpoint_file = CHECKPOINT_PATH / f"lgbm_checkpoint_{last_store}.pkl"

        if checkpoint_file.exists():
            print(f"Resuming from checkpoint: {checkpoint_file}")
            model = joblib.load(checkpoint_file)

        if features_file.exists():
            features = joblib.load(features_file)

        if categorical_file.exists():
            categorical_used = joblib.load(categorical_file)

print(f"Completed stores: {completed_stores}")

# =========================
# TRAIN LOOP
# =========================

for store in stores:

    if store in completed_stores:
        print(f"Skipping completed store: {store}")
        continue

    print(f"\nTraining on {store}")

    df = pd.read_parquet(
        DATA_PATH,
        filters=[("store_id", "==", store)]
    )

    df["date"] = pd.to_datetime(df["date"])

    train = df[df["date"] < "2015-01-01"]
    val = df[(df["date"] >= "2015-01-01") & (df["date"] < "2016-01-01")]

    if features is None:
        features = [
            c for c in df.columns
            if c not in ["sales", "date", "id", "d"]
        ]

        categorical_used = [
            c for c in categorical_cols
            if c in features
        ]

        joblib.dump(features, features_file)
        joblib.dump(categorical_used, categorical_file)

    X_train = train[features].copy()
    y_train = train["sales"].copy()

    X_val = val[features].copy()
    y_val = val["sales"].copy()

    for col in categorical_used:
        if col in X_train.columns:
            X_train[col] = X_train[col].astype("category")
            X_val[col] = X_val[col].astype("category")

    if model is None:
        model = lgb.LGBMRegressor(
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=96,
            max_depth=-1,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            objective="regression",
            n_jobs=-1,
            random_state=42,
            force_row_wise=True
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            categorical_feature=categorical_used,
            callbacks=[
                lgb.early_stopping(stopping_rounds=50),
                lgb.log_evaluation(period=50)
            ]
        )

    else:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            categorical_feature=categorical_used,
            init_model=model,
            callbacks=[
                lgb.early_stopping(stopping_rounds=50),
                lgb.log_evaluation(period=50)
            ]
        )

    # =========================
    # SAVE CHECKPOINT
    # =========================

    checkpoint_file = CHECKPOINT_PATH / f"lgbm_checkpoint_{store}.pkl"
    joblib.dump(model, checkpoint_file)

    completed_stores.append(store)

    pd.DataFrame({
        "store_id": completed_stores
    }).to_csv(progress_file, index=False)


 
    joblib.dump(features, features_file)
    joblib.dump(categorical_used, categorical_file)

    print(f"Checkpoint saved: {checkpoint_file}")

    del df, train, val, X_train, X_val, y_train, y_val
    gc.collect()

# =========================
# FINAL SAVE
# =========================

joblib.dump(model, MODEL_PATH / "lgbm_global.pkl")
joblib.dump(features, MODEL_PATH / "features_global.pkl")
joblib.dump(categorical_used, MODEL_PATH / "categorical_cols_global.pkl")

end_total = time.time()

print(f"\nTotal training time: {(end_total - start_total) / 60:.2f} min")
print("Global model saved")

Overwriting /kaggle/working/src/modeling/run_modelo_global.py


## 3.2 run_post_analysis

In [8]:
%%writefile /kaggle/working/src/modeling/run_post_analysis.py
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import time
import gc

start_total = time.time()

BASE_DIR = Path("/kaggle/working")

DATA_PATH = BASE_DIR / "data" / "features" / "m5_features"
MODEL_PATH = BASE_DIR / "models"
OUTPUT_PATH = BASE_DIR / "post_analysis"
PRED_PATH = OUTPUT_PATH / "predictions"

PRED_PATH.mkdir(parents=True, exist_ok=True)

model = joblib.load(MODEL_PATH / "lgbm_global.pkl")
features = joblib.load(MODEL_PATH / "features_global.pkl")
categorical_cols = joblib.load(MODEL_PATH / "categorical_cols_global.pkl")

stores = pd.read_parquet(
    DATA_PATH,
    columns=["store_id"]
)["store_id"].unique().tolist()

def prepare_X(df):
    X = df[features].copy()
    for col in categorical_cols:
        if col in X.columns:
            X[col] = X[col].astype("category")
    return X

for store in stores:

    output_file = PRED_PATH / f"pred_{store}.parquet"

    if output_file.exists():
        print(f"Skipping existing prediction: {store}")
        continue

    print(f"\nProcessing {store}")

    df = pd.read_parquet(
        DATA_PATH,
        filters=[("store_id", "==", store)]
    )

    df["date"] = pd.to_datetime(df["date"])

    X = prepare_X(df)

    df["prediction"] = model.predict(X)
    df["prediction"] = df["prediction"].clip(lower=0)

    df.to_parquet(
        output_file, 
        engine="pyarrow",
        coerce_timestamps="ms",
        allow_truncated_timestamps=True,
        index=False
    )



    print(f"Saved: {output_file}")

    del df, X
    gc.collect()

end_total = time.time()

print(f"\nTotal prediction time: {(end_total - start_total) / 60:.2f} min")
print("Predictions completed")

Overwriting /kaggle/working/src/modeling/run_post_analysis.py


## 3.3 shap_analysis 

In [9]:
%%writefile /kaggle/working/src/modeling/run_shap_analysis.py

import pandas as pd
import numpy as np
import joblib
import shap
import time
import gc
from pathlib import Path

start_total = time.time()

print("=== START SHAP ANALYSIS ===")

BASE_DIR = Path("/kaggle/working")
DATA_PATH = BASE_DIR / "data" / "features" / "m5_features"
MODEL_PATH = BASE_DIR / "models"
OUTPUT_PATH = BASE_DIR / "post_analysis"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("Loading model artifacts...")

model = joblib.load(MODEL_PATH / "lgbm_global.pkl")
features = joblib.load(MODEL_PATH / "features_global.pkl")
categorical_cols = joblib.load(MODEL_PATH / "categorical_cols_global.pkl")

print(f"Model loaded")
print(f"Features loaded: {len(features)}")
print(f"Categorical cols loaded: {len(categorical_cols)}")

stores = pd.read_parquet(
    DATA_PATH,
    columns=["store_id"]
)["store_id"].unique().tolist()

print(f"Stores detected: {stores}")

N_TOTAL = 50000
N_PER_STORE = max(1, N_TOTAL // len(stores))

print(f"Target total sample rows: {N_TOTAL}")
print(f"Rows per store: {N_PER_STORE}")

samples = []

cols_to_read = list(dict.fromkeys(features + ["store_id"]))

print(f"Columns to read: {len(cols_to_read)}")

for store in stores:

    print(f"\nLoading SHAP sample from {store}")

    start_store = time.time()

    df_store = pd.read_parquet(
        DATA_PATH,
        columns=cols_to_read,
        filters=[("store_id", "==", store)]
    )

    print(f"{store} rows loaded: {len(df_store)}")

    sample_store = df_store.sample(
        n=min(N_PER_STORE, len(df_store)),
        random_state=42
    )

    print(f"{store} sampled rows: {len(sample_store)}")

    samples.append(sample_store)

    del df_store
    gc.collect()

    print(f"{store} completed in {(time.time() - start_store):.2f} sec")

print("\nConcatenating samples...")

df_sample = pd.concat(samples, ignore_index=True)

print(f"Concatenated rows: {len(df_sample)}")

if len(df_sample) > N_TOTAL:

    print("Applying final global sample reduction...")

    df_sample = df_sample.sample(
        n=N_TOTAL,
        random_state=42
    )

print(f"Final SHAP sample rows: {len(df_sample)}")

X_sample = df_sample[features].copy()

print(f"X_sample shape: {X_sample.shape}")

for col in categorical_cols:
    if col in X_sample.columns:
        X_sample[col] = X_sample[col].astype("category")

print("Categorical conversion completed")

del df_sample
gc.collect()

print(f"\nComputing SHAP for {len(X_sample)} rows and {len(features)} features...")

start_shap = time.time()

explainer = shap.TreeExplainer(model)

print("TreeExplainer initialized")

shap_values = explainer.shap_values(X_sample)

print("SHAP computation completed")

print(f"SHAP compute time: {(time.time() - start_shap)/60:.2f} min")

print("Building SHAP dataframe...")

shap_df = pd.DataFrame({
    "feature": X_sample.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

shap_df["store_id"] = "GLOBAL"
shap_df["sample_rows"] = len(X_sample)
shap_df["sample_strategy"] = "stratified_by_store"
shap_df["n_features_model"] = len(features)

print("SHAP dataframe built")

output_file = OUTPUT_PATH / "shap_values.parquet"

print(f"Saving parquet to: {output_file}")

shap_df.to_parquet(output_file, index=False)

print(f"Saved: {output_file}")

print("\nTop 10 SHAP features:")

print(shap_df.head(10))

print(f"\nTotal SHAP pipeline time: {(time.time() - start_total)/60:.2f} min")

print("=== SHAP ANALYSIS COMPLETED ===")

Overwriting /kaggle/working/src/modeling/run_shap_analysis.py


## 3.4 build_hierarchy.py

In [10]:
%%writefile /kaggle/working/src/modeling/build_hierarchy.py
import glob
import pandas as pd
import time
from pathlib import Path
import gc

BASE_DIR = Path(__file__).resolve().parents[2]
OUTPUT = BASE_DIR / "post_analysis"
OUTPUT.mkdir(parents=True, exist_ok=True)

start_time = time.time()

PRED_PATH = BASE_DIR / "post_analysis" / "predictions"
files = glob.glob(str(PRED_PATH / "pred_*.parquet"))

print(f"Found {len(files)} files")

if len(files) == 0:
    raise ValueError("No prediction files found in predictions/")


def build_level(data, group_cols, level_name):
    g = (
        data
        .groupby(group_cols + ["date"], observed=True)[["prediction", "sales"]]
        .sum()
        .reset_index()
    )

    for col in group_cols:
        g[col] = g[col].astype(str)

    if len(group_cols) == 1:
        g["series_id"] = g[group_cols[0]]
    else:
        g["series_id"] = g[group_cols[0]]
        for col in group_cols[1:]:
            g["series_id"] += "_" + g[col]

    g["level"] = level_name

    return g[["series_id", "date", "prediction", "sales", "level"]]


store_blocks = []

for f in files:
    print(f"\nLoading {f}")

    df = pd.read_parquet(f)

    needed_cols = [
        "store_id", "item_id", "dept_id", "cat_id", "state_id",
        "date", "prediction", "sales"
    ]

    df = df[needed_cols].copy()

    for col in ["store_id", "item_id", "dept_id", "cat_id", "state_id"]:
        df[col] = df[col].astype(str)

    df["date"] = pd.to_datetime(df["date"])

    store_blocks.append(df)

full = pd.concat(store_blocks, ignore_index=True)

del store_blocks
gc.collect()

levels = []

levels.append(build_level(full, ["store_id", "item_id"], "item"))
levels.append(build_level(full, ["store_id", "dept_id"], "dept"))
levels.append(build_level(full, ["store_id", "cat_id"], "cat"))
levels.append(build_level(full, ["store_id"], "store"))
levels.append(build_level(full, ["state_id"], "state"))
levels.append(build_level(full, ["state_id", "cat_id"], "state_cat"))
levels.append(build_level(full, ["state_id", "dept_id"], "state_dept"))

total = (
    full
    .groupby(["date"], observed=True)[["prediction", "sales"]]
    .sum()
    .reset_index()
)

total["series_id"] = "total"
total["level"] = "total"

levels.append(total[["series_id", "date", "prediction", "sales", "level"]])

hierarchy = pd.concat(levels, ignore_index=True)

OUTPUT_FILE = OUTPUT / "hierarchy_predictions.parquet"
hierarchy.to_parquet(
           OUTPUT_FILE, 
           engine="pyarrow",
           coerce_timestamps="ms",
           allow_truncated_timestamps=True,
           index=False
)

# =========================
# SAVE BOTTOM-UP BASELINE
# =========================

BOTTOM_UP_FILE = OUTPUT / "bottom_up_predictions.parquet"

bottom_up = hierarchy.rename(columns={
    "prediction": "prediction_bottom_up"
})

bottom_up.to_parquet(
           BOTTOM_UP_FILE, 
           engine="pyarrow",
           coerce_timestamps="ms",
           allow_truncated_timestamps=True,
           index=False
)

print("Bottom-Up predictions saved successfully")
print(f"Bottom-Up output: {BOTTOM_UP_FILE}")
print(f"Bottom-Up rows: {bottom_up.shape[0]}")


end_time = time.time()

print(f"Time: {(end_time - start_time) / 60:.2f} minutes")
print("Hierarchy built successfully")
print(f"Rows: {hierarchy.shape[0]}")

Overwriting /kaggle/working/src/modeling/build_hierarchy.py


## 3.5 build_m5_evaluation.py

In [11]:
%%writefile /kaggle/working/src/modeling/build_m5_evaluation.py
import pandas as pd
import numpy as np
from pathlib import Path
import time

start_total = time.time()

BASE_DIR = Path(__file__).resolve().parents[2]

DATA_PATH = BASE_DIR / "data" / "raw"
OUTPUT_PATH = BASE_DIR / "post_analysis"
OUTPUT_PATH.mkdir(exist_ok=True)

print("Loading M5 raw data...")

sales = pd.read_csv(DATA_PATH / "sales_train_validation.csv")
calendar = pd.read_csv(DATA_PATH / "calendar.csv")
prices = pd.read_csv(DATA_PATH / "sell_prices.csv")

d_cols = [c for c in sales.columns if c.startswith("d_")]

df = sales.melt(
    id_vars=["item_id", "dept_id", "cat_id", "store_id", "state_id"],
    value_vars=d_cols,
    var_name="d",
    value_name="sales"
)

df = df.merge(calendar[["d", "date", "wm_yr_wk"]], on="d", how="left")
df = df.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

df["sell_price"] = df["sell_price"].fillna(0)
df["revenue"] = df["sales"] * df["sell_price"]
df["date"] = pd.to_datetime(df["date"])

train = df[df["date"] < "2016-01-01"]

print("Computing RMSSE scale...")

scale = (
    train
    .sort_values(["store_id", "item_id", "date"])
    .groupby(["item_id", "store_id"])["sales"]
    .apply(lambda x: np.mean(np.diff(x.values) ** 2) + 1e-8)
    .reset_index()
)

scale["series_id"] = scale["store_id"].astype(str) + "_" + scale["item_id"].astype(str)
scale = scale.rename(columns={"sales": "scale"})

scale.to_parquet(
       OUTPUT_PATH / "scale.parquet", 
       engine="pyarrow",
       coerce_timestamps="ms",
       allow_truncated_timestamps=True,
       index=False
)

print("Computing weights...")

weights = (
    train
    .groupby(["item_id", "store_id"])["revenue"]
    .sum()
    .reset_index()
)

weights["series_id"] = weights["store_id"].astype(str) + "_" + weights["item_id"].astype(str)

total_revenue = weights["revenue"].sum()

if total_revenue == 0:
    raise ValueError("Total revenue is zero. Cannot compute WRMSSE weights.")

weights["weight"] = weights["revenue"] / total_revenue

weights.to_parquet(
         OUTPUT_PATH / "weights.parquet", 
         engine="pyarrow",
         coerce_timestamps="ms",
         allow_truncated_timestamps=True,
         index=False
)

end_total = time.time()

print(f"\nTotal Evaluation data built time: {(end_total - start_total) / 60:.2f} min")
print("Evaluation data built")

Overwriting /kaggle/working/src/modeling/build_m5_evaluation.py


## 3.6 run_mint_top_revenue.py

In [12]:
%%writefile /kaggle/working/src/modeling/run_mint_top_revenue.py
import glob
import pandas as pd
import numpy as np
from pathlib import Path
import time
import gc

start_time = time.time()

BASE_DIR = Path(__file__).resolve().parents[2]

PRED_PATH = BASE_DIR / "post_analysis" / "predictions"
HIERARCHY_PATH = BASE_DIR / "post_analysis" / "hierarchy_predictions.parquet"
WEIGHTS_PATH = BASE_DIR / "post_analysis" / "weights.parquet"
OUTPUT_PATH = BASE_DIR / "post_analysis" / "mint_predictions.parquet"

TOP_N = 500
VALIDATION_START = "2015-01-01"
VALIDATION_END = "2016-01-01"
RIDGE = 1e-6

print("Loading hierarchy...")
hierarchy = pd.read_parquet(HIERARCHY_PATH)

hierarchy["date"] = pd.to_datetime(hierarchy["date"])
hierarchy["prediction_mint"] = hierarchy["prediction"]

print("Loading weights...")
weights = pd.read_parquet(WEIGHTS_PATH)

top_series = (
    weights
    .sort_values("weight", ascending=False)
    .head(TOP_N)["series_id"]
    .tolist()
)

print(f"Top series selected: {len(top_series)}")

print("Loading item-level predictions for selected series...")

files = glob.glob(str(PRED_PATH / "pred_*.parquet"))
parts = []

for f in files:
    df = pd.read_parquet(f)

    needed_cols = [
        "store_id", "item_id", "dept_id", "cat_id", "state_id",
        "date", "prediction", "sales"
    ]

    df = df[needed_cols].copy()

    for col in ["store_id", "item_id", "dept_id", "cat_id", "state_id"]:
        df[col] = df[col].astype(str)

    df["date"] = pd.to_datetime(df["date"])

    df["series_id"] = df["store_id"] + "_" + df["item_id"]

    df = df[df["series_id"].isin(top_series)]

    if len(df) > 0:
        parts.append(df)

    del df
    gc.collect()

bottom = pd.concat(parts, ignore_index=True)

del parts
gc.collect()

if bottom.empty:
    raise ValueError("No selected bottom-level series found.")

bottom_series = sorted(bottom["series_id"].unique().tolist())
n_bottom = len(bottom_series)

print(f"Bottom-level series available: {n_bottom}")

# =========================
# Construcción de nodos jerárquicos sobre top series
# =========================

meta = (
    bottom[["series_id", "store_id", "item_id", "dept_id", "cat_id", "state_id"]]
    .drop_duplicates("series_id")
    .set_index("series_id")
)

node_defs = []

node_defs.append(("total", "total", bottom_series))

for store, sids in meta.groupby("store_id").groups.items():
    node_defs.append((store, "store", list(sids)))

for state, sids in meta.groupby("state_id").groups.items():
    node_defs.append((state, "state", list(sids)))

for cat, sids in meta.groupby("cat_id").groups.items():
    node_defs.append((cat, "cat", list(sids)))

for dept, sids in meta.groupby("dept_id").groups.items():
    node_defs.append((dept, "dept", list(sids)))

for (state, cat), sids in meta.groupby(["state_id", "cat_id"]).groups.items():
    node_defs.append((f"{state}_{cat}", "state_cat", list(sids)))

for (state, dept), sids in meta.groupby(["state_id", "dept_id"]).groups.items():
    node_defs.append((f"{state}_{dept}", "state_dept", list(sids)))

for sid in bottom_series:
    node_defs.append((sid, "item", [sid]))

node_ids = [x[0] for x in node_defs]
n_nodes = len(node_ids)

print(f"Nodes in exact MinT block: {n_nodes}")

bottom_index = {sid: i for i, sid in enumerate(bottom_series)}

S = np.zeros((n_nodes, n_bottom), dtype=np.float64)

for row_idx, (_, _, children) in enumerate(node_defs):
    for sid in children:
        if sid in bottom_index:
            S[row_idx, bottom_index[sid]] = 1.0

# =========================
# Base forecasts y reales para nodos
# =========================

bottom_pred = bottom.pivot_table(
    index="date",
    columns="series_id",
    values="prediction",
    observed=True
).fillna(0)

bottom_true = bottom.pivot_table(
    index="date",
    columns="series_id",
    values="sales",
    observed=True
).fillna(0)

bottom_pred = bottom_pred.reindex(columns=bottom_series, fill_value=0)
bottom_true = bottom_true.reindex(columns=bottom_series, fill_value=0)

Y_bottom_hat = bottom_pred.values
Y_bottom_true = bottom_true.values

Y_hat_nodes = Y_bottom_hat @ S.T
Y_true_nodes = Y_bottom_true @ S.T

dates = bottom_pred.index

# =========================
# W: matriz de covarianza de errores en validación
# =========================

print("Computing W covariance matrix from validation residuals...")

val_mask = (dates >= pd.Timestamp(VALIDATION_START)) & (dates < pd.Timestamp(VALIDATION_END))

residuals = Y_true_nodes[val_mask, :] - Y_hat_nodes[val_mask, :]

if residuals.shape[0] < 2:
    raise ValueError("Not enough validation dates to estimate covariance matrix.")

W = np.cov(residuals, rowvar=False)

W = W + np.eye(W.shape[0]) * RIDGE

W_inv = np.linalg.pinv(W)

# =========================
# MinT exacto:
# y_rec = S (S' W^-1 S)^-1 S' W^-1 y_hat
# =========================

print("Applying exact MinT on selected top-revenue block...")

middle = np.linalg.pinv(S.T @ W_inv @ S)
P = S @ middle @ S.T @ W_inv

Y_rec_nodes = (P @ Y_hat_nodes.T).T
Y_rec_nodes = np.clip(Y_rec_nodes, 0, None)

mint_block = pd.DataFrame(
    Y_rec_nodes,
    index=dates,
    columns=node_ids
).reset_index().melt(
    id_vars="date",
    var_name="series_id",
    value_name="prediction_mint_exact"
)

# =========================
# Mezcla con pipeline completo
# =========================

print("Merging exact MinT block with full hierarchy...")

hierarchy = hierarchy.merge(
    mint_block,
    on=["date", "series_id"],
    how="left"
)

hierarchy["prediction_mint"] = hierarchy["prediction_mint_exact"].combine_first(
    hierarchy["prediction_mint"]
)

hierarchy = hierarchy.drop(columns=["prediction_mint_exact"])

hierarchy[["series_id", "date", "prediction_mint"]].to_parquet(
    OUTPUT_PATH,
    engine="pyarrow",
    coerce_timestamps="ms",
    allow_truncated_timestamps=True,
    index=False
)

end_time = time.time()

print(f"Exact selective MinT completed in {(end_time - start_time) / 60:.2f} minutes")
print(f"Output saved to: {OUTPUT_PATH}")

Overwriting /kaggle/working/src/modeling/run_mint_top_revenue.py


## 3.7 compute_wrmsse_final.py

In [13]:
%%writefile /kaggle/working/src/modeling/compute_wrmsse_final.py
import pandas as pd
import numpy as np
import time
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parents[2]
start_time = time.time()

print("Loading data...")

pred = pd.read_parquet(
    BASE_DIR / "post_analysis" / "mint_predictions.parquet",
    columns=["series_id", "date", "prediction_mint"]
)

true = pd.read_parquet(
    BASE_DIR / "post_analysis" / "hierarchy_predictions.parquet",
    columns=["series_id", "date", "sales", "level"]
)

scale = pd.read_parquet(BASE_DIR / "post_analysis" / "scale.parquet")
weights = pd.read_parquet(BASE_DIR / "post_analysis" / "weights.parquet")

pred["date"] = pd.to_datetime(pred["date"])
true["date"] = pd.to_datetime(true["date"])

# IMPORTANTE: no filtrar a item, porque MINT trae toda la jerarquía
true = true.copy()

df = pred.merge(
    true[["series_id", "date", "sales"]],
    on=["series_id", "date"],
    how="inner"
)

print("Merged rows:", len(df))

df = df.merge(scale[["series_id", "scale"]], on="series_id", how="inner")
df = df.merge(weights[["series_id", "weight"]], on="series_id", how="inner")

print("Rows after scale/weights:", len(df))

print("Computing RMSSE by series...")

df["sq_error"] = (df["sales"] - df["prediction_mint"]) ** 2

rmsse_df = (
    df
    .groupby("series_id", as_index=False)
    .agg(
        mse=("sq_error", "mean"),
        scale=("scale", "first")
    )
)

rmsse_df["rmsse"] = np.sqrt(rmsse_df["mse"] / rmsse_df["scale"])
rmsse_df = rmsse_df.replace([np.inf, -np.inf], np.nan)
rmsse_df = rmsse_df.dropna(subset=["rmsse"])
rmsse_df = rmsse_df[["series_id", "rmsse"]]

print("RMSSE rows:", len(rmsse_df))

final = rmsse_df.merge(weights[["series_id", "weight"]], on="series_id", how="inner")
final = final.dropna(subset=["rmsse", "weight"])

print("Final rows:", len(final))

final["wrmsse_component"] = final["rmsse"] * final["weight"]

wrmsse = final["wrmsse_component"].sum()
mean_rmsse = final["rmsse"].mean()

result_df = pd.DataFrame({
    "metric": ["WRMSSE", "MEAN_RMSSE"],
    "value": [wrmsse, mean_rmsse]
})

result_df.to_parquet(
    BASE_DIR / "post_analysis" / "wrmsse_result.parquet",
    engine="pyarrow",
    coerce_timestamps="ms",
    allow_truncated_timestamps=True,
    index=False
)

final.to_parquet(
    BASE_DIR / "post_analysis" / "wrmsse_details.parquet",
    engine="pyarrow",
    index=False
)

end_time = time.time()

print(f"\nWRMSSE calculated in {(end_time - start_time) / 60:.2f} minutes")
print("FINAL WRMSSE:", wrmsse)
print("MEAN RMSSE:", mean_rmsse)

Overwriting /kaggle/working/src/modeling/compute_wrmsse_final.py


## 3.8 LTSM 

In [14]:
%%writefile /kaggle/working/src/modeling/run_lstm_baseline.py
import pandas as pd
import numpy as np
from pathlib import Path
import time

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

start = time.time()

BASE_DIR = Path("/kaggle/working")
DATA_PATH = BASE_DIR / "data" / "features" / "m5_features"
OUT = BASE_DIR / "post_analysis" / "lstm"

OUT.mkdir(parents=True, exist_ok=True)

STORE_ID = "CA_1"
WINDOW = 28

print(f"Loading store {STORE_ID}")

df = pd.read_parquet(
    DATA_PATH,
    filters=[("store_id", "==", STORE_ID)],
    columns=["date", "sales"]
)

df["date"] = pd.to_datetime(df["date"])

# Serie agregada diaria por tienda
daily = (
    df.groupby("date", as_index=False)["sales"]
    .sum()
    .sort_values("date")
)

values = daily["sales"].values.reshape(-1, 1)

scaler = MinMaxScaler()
values_scaled = scaler.fit_transform(values)

def make_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i])
    return np.array(X), np.array(y)

X, y = make_sequences(values_scaled, WINDOW)

dates = daily["date"].iloc[WINDOW:].reset_index(drop=True)

train_mask = dates < pd.Timestamp("2015-01-01")
val_mask = (dates >= pd.Timestamp("2015-01-01")) & (dates < pd.Timestamp("2016-01-01"))

X_train = X[train_mask]
y_train = y[train_mask]

X_val = X[val_mask]
y_val = y[val_mask]

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)


model = Sequential([
    Input(shape=(WINDOW, 1)),
    LSTM(64),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse"
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

pred_scaled = model.predict(X_val)
pred = scaler.inverse_transform(pred_scaled)
y_true = scaler.inverse_transform(y_val)

rmse = np.sqrt(mean_squared_error(y_true, pred))
mae = mean_absolute_error(y_true, pred)

result = pd.DataFrame({
    "model": ["LSTM baseline"],
    "store_id": [STORE_ID],
    "window": [WINDOW],
    "rmse": [rmse],
    "mae": [mae],
    "time_min": [(time.time() - start) / 60]
})

pred_df = pd.DataFrame({
    "date": dates[val_mask].values,
    "sales_real": y_true.flatten(),
    "prediction_lstm": pred.flatten(),
    "store_id": STORE_ID
})

result.to_parquet(
        OUT / "lstm_metrics.parquet", 
        engine="pyarrow",
        coerce_timestamps="ms",
        allow_truncated_timestamps=True,
        index=False
)
pred_df.to_parquet(
         OUT / "lstm_predictions.parquet", 
         engine="pyarrow",
         coerce_timestamps="ms",
         allow_truncated_timestamps=True,
         index=False
)

model.save(OUT / "lstm_model.keras")


print(f"LSTM execution time: {(time.time() - start) / 60:.2f} min")

print(result)
print("LSTM baseline completed")

Overwriting /kaggle/working/src/modeling/run_lstm_baseline.py


In [15]:
%%writefile /kaggle/working/src/modeling/run_lstm_store_baseline.py
import pandas as pd
import numpy as np
from pathlib import Path
import time
import gc

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

start_total = time.time()

BASE_DIR = Path("/kaggle/working")
DATA_PATH = BASE_DIR / "data" / "features" / "m5_features"
OUT = BASE_DIR / "post_analysis" / "lstm_store"

OUT.mkdir(parents=True, exist_ok=True)

WINDOW = 28

stores = pd.read_parquet(
    DATA_PATH,
    columns=["store_id"]
)["store_id"].unique().tolist()

results = []
all_predictions = []

def make_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i])
    return np.array(X), np.array(y)

for store_id in stores:

    print(f"\nTraining LSTM for store: {store_id}")

    start_store = time.time()

    df = pd.read_parquet(
        DATA_PATH,
        filters=[("store_id", "==", store_id)],
        columns=["date", "sales"]
    )

    df["date"] = pd.to_datetime(df["date"])

    daily = (
        df.groupby("date", as_index=False)["sales"]
        .sum()
        .sort_values("date")
    )

    values = daily["sales"].values.reshape(-1, 1)

    scaler = MinMaxScaler()
    values_scaled = scaler.fit_transform(values)

    X, y = make_sequences(values_scaled, WINDOW)
    dates = daily["date"].iloc[WINDOW:].reset_index(drop=True)

    train_mask = dates < pd.Timestamp("2015-01-01")
    val_mask = (dates >= pd.Timestamp("2015-01-01")) & (dates < pd.Timestamp("2016-01-01"))

    X_train = X[train_mask]
    y_train = y[train_mask]

    X_val = X[val_mask]
    y_val = y[val_mask]

    model = Sequential([
        Input(shape=(WINDOW, 1)),
        LSTM(64),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    pred_scaled = model.predict(X_val, verbose=0)

    pred = scaler.inverse_transform(pred_scaled)
    y_true = scaler.inverse_transform(y_val)

    rmse = np.sqrt(mean_squared_error(y_true, pred))
    mae = mean_absolute_error(y_true, pred)

    time_min = (time.time() - start_store) / 60

    results.append({
        "model": "LSTM store-level baseline",
        "store_id": store_id,
        "window": WINDOW,
        "train_windows": len(X_train),
        "validation_windows": len(X_val),
        "epochs_ran": len(history.history["loss"]),
        "rmse": rmse,
        "mae": mae,
        "time_min": time_min
    })

    pred_df = pd.DataFrame({
        "date": dates[val_mask].values,
        "sales_real": y_true.flatten(),
        "prediction_lstm": pred.flatten(),
        "store_id": store_id
    })

    all_predictions.append(pred_df)

    print(f"{store_id} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | Time: {time_min:.2f} min")

    del df, daily, X, y, X_train, X_val, y_train, y_val, model
    tf.keras.backend.clear_session()
    gc.collect()

results_df = pd.DataFrame(results)
predictions_df = pd.concat(all_predictions, ignore_index=True)

results_df.to_parquet(
            OUT / "lstm_store_metrics.parquet", 
            engine="pyarrow",
            coerce_timestamps="ms",
            allow_truncated_timestamps=True,
            index=False
)
predictions_df.to_parquet(
                OUT / "lstm_store_predictions.parquet", 
                engine="pyarrow",
                coerce_timestamps="ms",
                allow_truncated_timestamps=True,
                index=False
)

total_time = (time.time() - start_total) / 60

summary = pd.DataFrame([{
    "model": "LSTM store-level baseline",
    "n_stores": len(stores),
    "window": WINDOW,
    "total_time_min": total_time,
    "mean_rmse": results_df["rmse"].mean(),
    "mean_mae": results_df["mae"].mean()
}])

summary.to_parquet(OUT / "lstm_store_summary.parquet", index=False)

print(f"LSTM store execution time: {total_time:.2f} min")

print("\n=== LSTM STORE BASELINE COMPLETED ===")
print(summary)
print(results_df)

Overwriting /kaggle/working/src/modeling/run_lstm_store_baseline.py


# 4 runners

## 4.1 run_all_spark.py

In [16]:
%%writefile /kaggle/working/run_all_spark.py
import sys
import time

sys.path.insert(0, "/kaggle/working")

from pathlib import Path
from src.spark.session import get_spark
from src.spark.load_data import load_data
from src.spark.validate import validate_data
from src.spark.build_features import build_features


def main():

    start = time.time()

    spark = get_spark()

    spark.conf.set("spark.sql.debug.maxToStringFields", 200)

    sales, calendar, prices = load_data(
        spark,
        path="/kaggle/working/data/raw"
    )

    validate_data(sales, calendar, prices)

    df = build_features(sales, calendar, prices)

    output_path = "/kaggle/working/data/features/m5_features"

    df.write.mode("overwrite").partitionBy("store_id").parquet(output_path)

    end = time.time()
    minutes = (end - start) / 60

    print("\n=== SPARK PIPELINE COMPLETED ===")
    print(f"OUTPUT PATH: {output_path}")
    print(f"TOTAL TIME: {minutes:.2f} min")

    path = Path(output_path)

    print("\nFILES GENERATED:")
    for f in path.glob("*"):
        print(f.name)

    spark.stop()


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/run_all_spark.py


## 4.2 run_modeling_pipeline.py

In [17]:
%%writefile /kaggle/working/run_all_modeling.py
import subprocess
import sys
import time

PYTHONPATH = "PYTHONPATH=/kaggle/working"

timings = []

def run_step(cmd, name):
    print(f"\n=== {name} ===")

    start = time.time()
    result = subprocess.run(f"{PYTHONPATH} {cmd}", shell=True)
    end = time.time()

    minutes = (end - start) / 60
    timings.append((name, minutes))

    print(f"\n{name} time: {minutes:.2f} min")

    if result.returncode != 0:
        print(f"\nERROR en {name}")
        sys.exit(1)

run_step("python /kaggle/working/src/modeling/run_modelo_global.py", "MODEL TRAINING")
run_step("python /kaggle/working/src/modeling/run_post_analysis.py", "POST ANALYSIS")
run_step("python /kaggle/working/src/modeling/run_shap_analysis.py", "SHAP")
run_step("python /kaggle/working/src/modeling/build_hierarchy.py", "BUILD HIERARCHY")
run_step("python /kaggle/working/src/modeling/build_m5_evaluation.py", "BUILD M5 EVALUATION")
run_step("python /kaggle/working/src/modeling/run_mint_top_revenue.py", "MINT TOP REVENUE")
run_step("python /kaggle/working/src/modeling/compute_wrmsse_final.py", "COMPUTE WRMSSE")

print("\n=== MODELING PIPELINE COMPLETED ===")
print("\n=== TIMING SUMMARY ===")

total = 0
for name, minutes in timings:
    total += minutes
    print(f"{name}: {minutes:.2f} min")

print(f"TOTAL MODELING TIME: {total:.2f} min")

Overwriting /kaggle/working/run_all_modeling.py


## 4.3 run_all_ltsm.py

In [18]:
%%writefile /kaggle/working/run_all_ltsm.py

import time
import subprocess

total_start = time.time()


def run_step(command, name):

    start = time.time()

    print(f"\n=== {name} ===\n")

    result = subprocess.run(command, shell=True)

    end = time.time()

    elapsed_min = (end - start) / 60

    print(f"\n{name} time: {elapsed_min:.2f} min")

    return elapsed_min


lstm_baseline_time = run_step(
    "python /kaggle/working/src/modeling/run_lstm_baseline.py",
    "LSTM BASELINE"
)

lstm_store_time = run_step(
    "python /kaggle/working/src/modeling/run_lstm_store_baseline.py",
    "LSTM STORE BASELINE"
)

total_end = time.time()

total_time = (total_end - total_start) / 60

print("\n=== ALL LSTM PIPELINES COMPLETED ===")
print(f"Total execution time: {total_time:.2f} min")

Overwriting /kaggle/working/run_all_ltsm.py


In [19]:
import os
import sys

print("cwd:", os.getcwd())
print("exists /kaggle/working/src:", os.path.exists("/kaggle/working/src"))
print("sys.path:")
for p in sys.path:
    print(p)

cwd: /kaggle/working
exists /kaggle/working/src: True
sys.path:
/kaggle/working
/kaggle/lib/kagglegym
/kaggle/lib
/usr/lib/python312.zip
/usr/lib/python3.12
/usr/lib/python3.12/lib-dynload

/usr/local/lib/python3.12/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.12/dist-packages/IPython/extensions
/root/.ipython


In [20]:
!touch /kaggle/working/src/__init__.py
!touch /kaggle/working/src/spark/__init__.py
!touch /kaggle/working/src/modeling/__init__.py

# 5. Pipeline execution

## 5.1 run_spark_pipeline

In [21]:
!python /kaggle/working/run_all_spark.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 02:36:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/12 02:37:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
Sales rows: 30490                                                               
Calendar rows: 1969
Prices rows: 6841121
Sales sin duplicados: 30490                                                     
Duplicados detectados: 0
26/05/12 02:38:16 WARN CacheManager: Asked to cache already cached data.
                                                                                
=== SPARK PIPELINE COMPLETED ===
OUTPUT PATH: /kaggle/working/data/features

## 5.2 run_all_modeling

In [ ]:
!python /kaggle/working/run_all_modeling.py


=== MODEL TRAINING ===
Completed stores: []

Training on CA_1
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Total Bins 6321
[LightGBM] [Info] Number of data points in the train set: 4369217, number of used features: 33
[LightGBM] [Info] Start training from score 1.277963
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.1751	valid_0's l2: 4.73107
[100]	valid_0's rmse: 2.05281	valid_0's l2: 4.21403
[150]	valid_0's rmse: 2.04027	valid_0's l2: 4.16269
[200]	valid_0's rmse: 2.03588	valid_0's l2: 4.14482
[250]	valid_0's rmse: 2.03415	valid_0's l2: 4.13777
[300]	valid_0's rmse: 2.03243	valid_0's l2: 4.13078
[350]	valid_0's rmse: 2.03203	valid_0's l2: 4.12913
[400]	valid_0's rmse: 2.03161	valid_0's l2: 4.12742
[450]	valid_0's rmse: 2.03153	valid_0's l2: 4.127

## 5.4 run_all_ltsm

In [ ]:
!python /kaggle/working/run_all_ltsm.py

# 6. Snapshot experimento

In [ ]:
# =========================
# GUARDAR SNAPSHOT COMPLETO
# =========================

!mkdir -p /kaggle/working/experiments/exp01_29f_baseline

!cp -r /kaggle/working/src /kaggle/working/experiments/exp01_29f_baseline/
!cp -r /kaggle/working/models /kaggle/working/experiments/exp01_29f_baseline/
!cp -r /kaggle/working/post_analysis /kaggle/working/experiments/exp01_29f_baseline/

print("Snapshot completed")

print("\nFiles in experiment folder:\n")

!ls -lh /kaggle/working/experiments/exp01_29f_baseline

In [ ]:
# Verificación
from pathlib import Path

exp_path = Path("/kaggle/working/experiments/exp01_29f_baseline")

summary = {
    "src_exists": (exp_path / "src").exists(),
    "models_exists": (exp_path / "models").exists(),
    "post_analysis_exists": (exp_path / "post_analysis").exists(),
    "metadata_exists": (exp_path / "metadata.json").exists(),
}

print(summary)

In [ ]:
!ls -lh /kaggle/working/experiments/exp01_29f_baseline/post_analysis/lstm

# 7. Verification / Results tables for report

## 7.1 RESUMEN DE FICHEROS GENERADOS

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/kaggle/working")  # o Path("../../") en local
POST = BASE / "post_analysis"
MODELS = BASE / "models"
FEATURES = BASE / "data" / "features" / "m5_features"

summary_files = []

for name, path in {
    "features": FEATURES,
    "models": MODELS,
    "post_analysis": POST,
    "predictions": POST / "predictions"
}.items():
    if path.exists():
        summary_files.append({
            "component": name,
            "path": str(path),
            "exists": True,
            "num_files": len(list(path.rglob("*")))
        })
    else:
        summary_files.append({
            "component": name,
            "path": str(path),
            "exists": False,
            "num_files": 0
        })

pd.DataFrame(summary_files)

## 7.2 NÚMERO DE FEATURES

In [ ]:
import joblib

features = joblib.load("/kaggle/working/models/features_global.pkl")

print("Training features:", len(features))
print(features)

## 7.3 Timings Kaggle 

SHAP global importance

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top_shap["feature"], top_shap["mean_abs_shap"])
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Feature")
plt.title("Importancia global de variables mediante SHAP")
plt.tight_layout()
plt.savefig(POST / "figura_n2_shap_importance.png", dpi=300, bbox_inches="tight")
plt.show()

## 7.4 WRMSSE final Kaggle

In [ ]:
wrmsse_result = pd.read_parquet(POST / "wrmsse_result.parquet")
wrmsse_result

## 7.5 Comparación local vs Kaggle-1

In [ ]:
compare_runs = pd.DataFrame([
    {
        "run": "Local initial validation",
        "environment": "Local i5 / 32 GB RAM",
        "features": 25,
        "reconciliation": "MinT diagonal approximation",
        "wrmsse": 1.3021,
        "total_time_min": 53.49
    },
    {
        "run": "Kaggle reproducible execution",
        "environment": "Kaggle CPU",
        "features": 29,
        "reconciliation": "Selective exact MinT on top revenue series",
        "wrmsse": 0.7683,
        "total_time_min": 287.88
    }
])

compare_runs

## 7.6 Mejora porcentual WRMSSE

In [ ]:
local_wrmsse = 1.3021
kaggle_wrmsse = 0.7682894510673492

improvement_pct = (local_wrmsse - kaggle_wrmsse) / local_wrmsse * 100

pd.DataFrame([{
    "local_wrmsse": local_wrmsse,
    "kaggle_wrmsse": kaggle_wrmsse,
    "improvement_pct": improvement_pct
}])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

wrmsse_compare = pd.DataFrame({
    "experiment": [
        "Local validation",
        "exp01_29f",
        "exp02_33f"
    ],
    "wrmsse": [
        1.3021,
        0.7682887196791824,
        0.75   # <- cambia por el valor real
    ]
})

plt.figure(figsize=(8, 5))

plt.bar(
    wrmsse_compare["experiment"],
    wrmsse_compare["wrmsse"]
)

plt.ylabel("WRMSSE")
plt.xlabel("Experiment")
plt.title("Figura N22. Evolución del WRMSSE entre experimentos")

plt.tight_layout()
plt.show()

## 7.7. Predicciones generadas por tienda


In [ ]:
pred_files = sorted((POST / "predictions").glob("pred_*.parquet"))

pred_summary = []

for f in pred_files:
    store = f.stem.replace("pred_", "")
    df = pd.read_parquet(f, columns=["date", "prediction", "sales"])
    pred_summary.append({
        "store_id": store,
        "rows": len(df),
        "date_min": df["date"].min(),
        "date_max": df["date"].max(),
        "prediction_mean": df["prediction"].mean(),
        "sales_mean": df["sales"].mean()
    })

pred_summary = pd.DataFrame(pred_summary)
pred_summary

## 7.8 Jerarquía generada


In [ ]:
hierarchy = pd.read_parquet(
    POST / "hierarchy_predictions.parquet",
    columns=["series_id", "date", "prediction", "sales", "level"]
)

hierarchy_summary = (
    hierarchy
    .groupby("level")
    .agg(
        rows=("series_id", "size"),
        n_series=("series_id", "nunique"),
        prediction_sum=("prediction", "sum"),
        sales_sum=("sales", "sum")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)
print(hierarchy_summary)

## 7.9 Coherencia total antes de reconciliación


In [ ]:
base_total = (
    hierarchy[hierarchy["level"] == "item"]
    .groupby("date")["prediction"]
    .sum()
    .sum()
)

hier_total = (
    hierarchy[hierarchy["level"] == "total"]["prediction"]
    .sum()
)

pd.DataFrame([{
    "item_sum_prediction": base_total,
    "total_level_prediction": hier_total,
    "absolute_difference": abs(base_total - hier_total)
}])

## 7.10 MinT selectivo: descripción cuantitativa


In [ ]:
mint = pd.read_parquet(POST / "mint_predictions.parquet")

mint_summary = pd.DataFrame([{
    "rows": len(mint),
    "n_series": mint["series_id"].nunique(),
    "date_min": mint["date"].min(),
    "date_max": mint["date"].max(),
    "prediction_mint_mean": mint["prediction_mint"].mean(),
    "prediction_mint_sum": mint["prediction_mint"].sum()
}])

mint_summary

## 7.11 Bottom Up

In [ ]:
import pandas as pd
from pathlib import Path

POST = Path("/kaggle/working/post_analysis")

bottom_up = pd.read_parquet(
    POST / "bottom_up_predictions.parquet",
    columns=["series_id", "date", "prediction_bottom_up", "sales", "level"]
)

bottom_up_summary = (
    bottom_up
    .groupby("level")
    .agg(
        rows=("series_id", "size"),
        n_series=("series_id", "nunique"),
        prediction_bottom_up_sum=("prediction_bottom_up", "sum"),
        sales_sum=("sales", "sum")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

bottom_up_summary

## 7.12 Bottom-Up vs MinT

In [ ]:
mint = pd.read_parquet(
    POST / "mint_predictions.parquet",
    columns=["series_id", "date", "prediction_mint"]
)

compare_bu_mint = bottom_up.merge(
    mint,
    on=["series_id", "date"],
    how="inner"
)

compare_bu_mint["diff_mint_vs_bu"] = (
    compare_bu_mint["prediction_mint"]
    - compare_bu_mint["prediction_bottom_up"]
)

summary_bu_mint = (
    compare_bu_mint
    .groupby("level")
    .agg(
        rows=("series_id", "size"),
        mean_abs_diff=("diff_mint_vs_bu", lambda x: x.abs().mean()),
        max_abs_diff=("diff_mint_vs_bu", lambda x: x.abs().max()),
        sum_bottom_up=("prediction_bottom_up", "sum"),
        sum_mint=("prediction_mint", "sum")
    )
    .reset_index()
)

summary_bu_mint

## 7.13 Evolución temporal real vs predicción

In [ ]:
bottom_up = pd.read_parquet(
    POST / "bottom_up_predictions.parquet",
    columns=["series_id", "date", "prediction_bottom_up", "sales", "level"]
)

mint = pd.read_parquet(
    POST / "mint_predictions.parquet",
    columns=["series_id", "date", "prediction_mint"]
)

compare_bu_mint = bottom_up.merge(
    mint,
    on=["series_id", "date"],
    how="inner"
)

compare_bu_mint["diff_mint_vs_bottom_up"] = (
    compare_bu_mint["prediction_mint"]
    - compare_bu_mint["prediction_bottom_up"]
)

summary_bu_mint = (
    compare_bu_mint
    .groupby("level")
    .agg(
        rows=("series_id", "size"),
        mean_abs_diff=("diff_mint_vs_bottom_up", lambda x: x.abs().mean()),
        max_abs_diff=("diff_mint_vs_bottom_up", lambda x: x.abs().max()),
        bottom_up_sum=("prediction_bottom_up", "sum"),
        mint_sum=("prediction_mint", "sum")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

summary_bu_mint

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path("/kaggle/working")
POST = BASE_DIR / "post_analysis"

weights = pd.read_parquet(POST / "weights.parquet")
bottom_up = pd.read_parquet(POST / "hierarchy_predictions.parquet")
mint = pd.read_parquet(POST / "mint_predictions.parquet")

top500_series = (
    weights
    .sort_values("weight", ascending=False)
    .head(500)["series_id"]
    .tolist()
)

bottom_top500 = (
    bottom_up[bottom_up["series_id"].isin(top500_series)]
    .merge(
        mint,
        on=["series_id", "date"],
        how="left"
    )
)

block_plot = (
    bottom_top500
    .groupby("date", as_index=False)
    .agg(
        sales=("sales", "sum"),
        prediction_bottom_up=("prediction", "sum"),
        prediction_mint=("prediction_mint", "sum")
    )
)

block_plot["date"] = pd.to_datetime(block_plot["date"])
block_plot = block_plot.sort_values("date")

block_plot["sales_smooth"] = block_plot["sales"].rolling(7).mean()
block_plot["base_smooth"] = block_plot["prediction_bottom_up"].rolling(7).mean()
block_plot["mint_smooth"] = block_plot["prediction_mint"].rolling(7).mean()

plt.figure(figsize=(14, 6))

plt.plot(
    block_plot["date"],
    block_plot["sales_smooth"],
    label="Ventas reales",
    linewidth=2
)

plt.plot(
    block_plot["date"],
    block_plot["base_smooth"],
    label="Bottom-Up top 500",
    linewidth=3.0,
    linestyle=":"
)

plt.plot(
    block_plot["date"],
    block_plot["mint_smooth"],
    label="MinT top 500",
    linewidth=1.0,
    alpha=0.9
)

plt.xlabel("Fecha")
plt.ylabel("Ventas")

plt.title(
    "Evolución temporal real vs Bottom-Up vs MinT "
    "en bloque top 500 revenue"
)

plt.legend()
plt.tight_layout()
plt.show()

## 7.14 WRMSSE details: top series con más impacto


In [ ]:
wrmsse_details = pd.read_parquet(POST / "wrmsse_details.parquet")

top_wrmsse = (
    wrmsse_details
    .sort_values("wrmsse_component", ascending=False)
    .head(10)
)

top_wrmsse

## 7.15 WRMSSE details: top series con más impacto BARRAS

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

wrmsse_details = pd.read_parquet(POST / "wrmsse_details.parquet")

top_wrmsse = (
    wrmsse_details
    .sort_values("wrmsse_component", ascending=False)
    .head(10)
)

top_wrmsse = top_wrmsse.sort_values("wrmsse_component")

plt.figure(figsize=(10, 6))

plt.barh(
    top_wrmsse["series_id"],
    top_wrmsse["wrmsse_component"]
)

plt.xlabel("WRMSSE component")
plt.ylabel("Series")
plt.title("Top 10 series contributing to WRMSSE")

plt.tight_layout()
plt.show()

## 7.16 Exploración LTSM

In [ ]:
import pandas as pd
from pathlib import Path

POST = Path("/kaggle/working/post_analysis")

lstm_metrics = pd.read_parquet(POST / "lstm" / "lstm_metrics.parquet")
lstm_predictions = pd.read_parquet(POST / "lstm" / "lstm_predictions.parquet")

display(lstm_metrics)
display(lstm_predictions.head())

## 7.17 Exploración LTSM por store

In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working/post_analysis/lstm_store")

lstm_summary = pd.read_parquet(OUT / "lstm_store_summary.parquet")
lstm_metrics = pd.read_parquet(OUT / "lstm_store_metrics.parquet")

display(lstm_summary)
display(lstm_metrics.sort_values("rmse"))

## 7.18 EMatriz de correlación

In [ ]:
# ============================================================
# MATRIZ DE CORRELACIÓN DE RESIDUOS
# MinT selectivo top revenue
# ============================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# 1. Cargar predicciones jerárquicas
# ============================================================

df = pd.read_parquet(
    "/kaggle/working/post_analysis/hierarchy_predictions.parquet"
)

# ============================================================
# 2. Filtrar únicamente series item
# ============================================================

df = df[df["level"] == "item"].copy()

# ============================================================
# 3. Construcción de residuos
# ============================================================

df["residual"] = df["sales"] - df["prediction"]

# ============================================================
# 4. Seleccionar top revenue series
#    (subconjunto utilizado en MinT exacto)
# ============================================================

TOP_N = 20

top_series = (
    df.groupby("series_id")["sales"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N)
    .index
)

df_top = df[df["series_id"].isin(top_series)]

# ============================================================
# 5. Pivot temporal de residuos
# ============================================================

residual_matrix = df_top.pivot(
    index="date",
    columns="series_id",
    values="residual"
)

# ============================================================
# 6. Matriz de correlación
# ============================================================

corr_matrix = residual_matrix.corr()

# ============================================================
# 7. Heatmap
# ============================================================

plt.figure(figsize=(16, 14))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"}
)

plt.title(
    "Figura X. Matriz de correlación de residuos del subconjunto "
    "top revenue utilizado en reconciliación MinT selectiva",
    fontsize=14,
    pad=20
)

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# EXPORTAR HEATMAP MINt
# ============================================================

OUTPUT_PATH = (
    "/kaggle/working/post_analysis/"
    "figura_mint_residual_correlation_heatmap.png"
)

plt.figure(figsize=(16, 14))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"}
)

plt.title(
    "Matriz de correlación de residuos - "
    "MinT selectivo top revenue",
    fontsize=14,
    pad=20
)

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()

# ============================================================
# EXPORTACIÓN
# ============================================================

plt.savefig(
    OUTPUT_PATH,
    dpi=300,
    bbox_inches="tight"
)



print(f"Figura exportada en: {OUTPUT_PATH}")

# 8. Descarga del proyecto

## 8.1 Descarga de ficheros de origen para ETL de integración de Power  BI Lake

In [ ]:
!zip -j powerbi_input.zip \
experiments/exp01_local/mint_predictions.parquet \
experiments/exp01_local/shap_values.parquet \
experiments/exp01_local/weights.parquet \
experiments/exp01_local/wrmsse_details.parquet \
experiments/exp01_local/wrmsse_result.parquet \
experiments/exp01_local/hierarchy_predictions.parquet

In [ ]:
from IPython.display import FileLink
FileLink(r'powerbi_input.zip')

## 8.2 Descarga del Proyecto completo (opcional, no recomendado por su elevado tamaño)

In [ ]:
%cd /kaggle/working

!zip -r f33_full.zip . \
-x "*.ipynb_checkpoints*" \
-x "*.git*" \
-x "__pycache__/*"

In [ ]:
from IPython.display import FileLink
FileLink(r'f33_full.zip')